# 6 WorkFlow Gerencial, futuro=SEP

### 6.1 Objetivo

Presentar un workflow/pipeline completo al que los estudiantes deberán enriquecer

#### 6.2  Seteo del ambiente en Google Colab

Esta parte se debe correr con el runtime en Python3
<br>Ir al menu, Runtime -> Change Runtime Type -> Runtime type ->  **Python 3**

Conectar la virtual machine donde esta corriendo Google Colab con el  Google Drive, para poder tener persistencia de archivos

In [1]:
# primero establecer el Runtime de Python 3
from google.colab import drive
drive.mount('/content/.drive')

Mounted at /content/.drive


Para correr la siguiente celda es fundamental en Arranque en Frio haber copiado el archivo kaggle.json al Google Drive, en la carpeta indicada en el instructivo

<br>los siguientes comando estan en shell script de Linux
*   Crear las carpetas en el Google Drive
*   "instalar" el archivo kaggle.json desde el Google Drive a la virtual machine para que pueda ser utilizado por la libreria  kaggle de Python
*   Bajar el  **dataset_pequeno**  al  Google Drive  y tambien al disco local de la virtual machine que esta corriendo Google Colab
*   Bajar el **dataset_historico** al Google Drive y tambien al disco local de la virtual machine que esta corriendo Google Colab



In [2]:
%%shell

mkdir -p "/content/.drive/My Drive/dmeyf"
mkdir -p "/content/buckets"
ln -sfn "/content/.drive/My Drive/dmeyf"   /content/buckets/b1

mkdir -p ~/.kaggle
cp /content/buckets/b1/kaggle/kaggle.json  ~/.kaggle
chmod 600 ~/.kaggle/kaggle.json


mkdir -p /content/buckets/b1/exp
mkdir -p /content/buckets/b1/datasets
mkdir -p /content/datasets


# defino funcion descargar()
descargar() {
  carpeta_destino="/content/buckets/b1/datasets/"
  url_origen="https://storage.googleapis.com/open-courses/utn2026-b40a/"
  archivo="$1"

  if ! test -f "$carpeta_destino""$archivo"; then
    wget  "$url_origen""$archivo"  -O "$carpeta_destino""$archivo"
  fi

  if ! test -f  "/content/datasets/""$archivo"; then
    cp  "$carpeta_destino""$archivo"  "/content/datasets/""$archivo"
  fi;
}


# hago la descarga efectiva, llamando a descargar()
descargar  "dataset_pequeno.csv"
descargar  "gerencial_competencia_2026.csv.gz"


## 6.3  Workflow

## Inicializacion

Esta parte se debe correr con el runtime en lenguaje **R** Ir al menu, Runtime -> Change Runtime Type -> Runtime type -> R

limpio el ambiente de R

In [31]:
format(Sys.time(), "%a %b %d %X %Y")

[1] "Thu Sep 17 12:19:00 AM 2026"

In [32]:
# limpio la memoria
rm(list=ls(all.names=TRUE)) # remove all objects
gc(full=TRUE, verbose=FALSE) # garbage collection

,used,(Mb),gc trigger,(Mb),max used,(Mb)
Ncells,1917380,102.4,3363074,179.7,3363074,179.7
Vcells,3502942,26.8,18352935,140.1,19050974,145.4


In [33]:
require("data.table")

if( !require("R.utils")) install.packages("R.utils")
require("R.utils")

#### Parametros
Si es gerente, no cambie nada
<br>Si es Analista, cambie el nombre del dataset

In [34]:
# Se opta por la recomendación del grupo A Problema 4: trabajar con el dataset completo
PARAM <- list()
PARAM$semilla_primigenia <- 100003
# Semillas Mariela: *100003  100049  100057  100069  100099

# Se opta por la recomendación del grupo A Problema 10: Feature Engineering Intra-mes Algoritmo genético
# Configuración del Algoritmo Genético de Feature Engineering Intra-mes (gramEvol)
PARAM$GA <- list(
  popSize = 150,            # Tamaño de la población de individuos
  iterations = 50,          # Cantidad de generaciones evolutivas
  top_features = 20,        # Cantidad de mejores features no lineales a inyectar al dataset
  max_terminales = 60,      # Forzar a usar todas las variables
  seqLen = 250,             # Longitud máxima de codones del genoma
  max.depth = 10,           # Profundidad máxima del árbol sintáctico BNF
  max_filas_fitness = 50000 # Muestra máxima para evaluación de fitness ultrarrápida
)

PARAM$experimento <- 6300
PARAM$dataset <- "gerencial_competencia_2026.csv.gz"

#### Carpeta del Experimento

In [35]:
# carpeta de trabajo

setwd("/content/buckets/b1/exp")
experimento_folder <- paste0("WF", PARAM$experimento)
dir.create(experimento_folder, showWarnings=FALSE)
dir_experimento_base <- paste0("/content/buckets/b1/exp/", experimento_folder)
setwd( dir_experimento_base )


### 6.3.1   Preprocesamiento del dataset

#### 6.3.1.1  DT incorporar dataset

In [36]:
# lectura del dataset
dataset <- fread(paste0("/content/datasets/", PARAM$dataset))

#### 6.3.1.2  CA  Catastrophe Analysis
Se intentan reparar las variables que para un mes están con todos los valores en cero.

El método que se utiliza es **Machine Learning** se asigna NA also valores, si ha leido bien, es la "anti imputación de valores faltantes"
<br> Usted podrá aplicar aquí otros métodos

In [37]:
# las pruebas no demostraron diferencias significativas, se deja el baseline
dataset[ foto_mes==202006, internet:=NA]
dataset[ foto_mes==202006, mrentabilidad:=NA]
dataset[ foto_mes==202006, mrentabilidad_annual:=NA]
dataset[ foto_mes==202006, mcomisiones:=NA]
dataset[ foto_mes==202006, mactivos_margen:=NA]
dataset[ foto_mes==202006, mpasivos_margen:=NA]
dataset[ foto_mes==202006, mcuentas_saldo:=NA]
dataset[ foto_mes==202006, ctarjeta_visa_transacciones:=NA]
dataset[ foto_mes==202006, mtarjeta_visa_consumo:=NA]
dataset[ foto_mes==202006, mtarjeta_master_consumo:=NA]
dataset[ foto_mes==202006, ccallcenter_transacciones:=NA]
dataset[ foto_mes==202006, chomebanking_transacciones:=NA]


#### 6.3.1.3  DR  Data Drifting
Se intenta corregir el data drifting, quizas ajustando por IPC ...
<br>Esta parte podrá ser abordada por todos los Analistas y también la Gerenciapero se decide pedagogicamente no incluirla en esta primer version para reducir la carga cognitiva

In [38]:
# Problema 02 - sin codigo en esta primera version del workflow


#### 6.3.1.3  FE_intra_manual Feature Engineering intra-mes

Agrego campos nuevos dentro del mismo mes, SIN considerar la historia.

In [39]:
# Problema 03 - se opta por la recomendación del grupo A: mantener el original como baseline
# esta funcion atributos presentes existe debido a que las modalidades poseen datasets con distinta cantidad de campos
atributos_presentes <- function( patributos )
{
  atributos <- unique( patributos )
  comun <- intersect( atributos, colnames(dataset) )

  return(  length( atributos ) == length( comun ) )
}

# el mes 1,2, ..12
if( atributos_presentes( c("foto_mes") ))
  dataset[, kmes := foto_mes %% 100]

# variable extraida de una tesis de maestria de Irlanda
if( atributos_presentes( c("mpayroll", "cliente_edad") ))
  dataset[, mpayroll_sobre_edad := mpayroll / cliente_edad]


9.3.1.3.2 FE_intra_GA: Feature Engineering Intra-mes mediante Algoritmo Genético (gramEvol)
Problema #10: Búsqueda Heurística Evolutiva de Combinaciones No Lineales
A continuación se implementa la evolución gramatical mediante el paquete gramEvol:

Población Terminal: El pool de variables candidatas se extrae de las columnas numéricas del dataset tras Catastrophe Analysis y Data Drifting.
Gramática Formal BNF: Genera árboles sintácticos estocásticos con operadores aritméticos protegidos (+, -, *, protected_div, protected_log_diff).
Fitness Basado en LightGBM Univariado Real: Cada cromosoma se evalúa mediante un modelo LightGBM entrenado sobre un split local (sin contaminar el mes de validación oficial 202107).
Inyección de Top N Features (GA_Feature_1 a GA_Feature_N): Las mejores combinaciones no triviales ni duplicadas descubiertas de forma autónoma por la evolución se evalúan sobre la totalidad del dataset.
Blindaje Estricto Anti-Leakage: Las etiquetas para el fitness se calculan exclusivamente en vectores aislados en memoria (y_tr_GA, y_val_GA), y se asegura que clase01 no quede en dataset antes de FEhist.

In [40]:
# ==============================================================================
# 9.3.1.3.2 FE_intra_GA: Algoritmo Genético (Grammatical Evolution)
# ==============================================================================

if (!require("gramEvol")) {
  install.packages("gramEvol", repos = "https://cloud.r-project.org", dependencies = TRUE)
}
if (!require("gramEvol")) {
  install.packages("gramEvol", repos = "https://cran.rstudio.com", dependencies = TRUE)
}
if( !require("lightgbm")) install.packages("lightgbm")
require("lightgbm")

require("gramEvol")
require("lightgbm")
require("data.table")
require("parallel")

if (is.null(PARAM$GA)) {
  PARAM$GA <- list(
    popSize = 150,
    iterations = 50,
    top_features = 5,
    max_terminales = 60,
    seqLen = 250,
    max.depth = 10,
    max_filas_fitness = 50000
  )
}

# 1. Variables candidatas para el Algoritmo Genético
excluir_GA <- c("numero_de_cliente", "foto_mes", "clase_ternaria", "clase01", "azar", "fold_train", "fold_final_train")
candidatas_GA <- setdiff(colnames(dataset), excluir_GA)

# Filtrar únicamente variables numéricas
son_numericas <- sapply(dataset[, candidatas_GA, with = FALSE], is.numeric)
candidatas_GA <- candidatas_GA[son_numericas]

# Excluir fechas
patron_fechas <- "^(f|.*_f|.*fecha)"
candidatas_GA <- candidatas_GA[!grepl(patron_fechas, candidatas_GA, ignore.case = TRUE)]

# Límite de seguridad de terminales para controlar el espacio de búsqueda
MAX_TERMINALES <- if (!is.null(PARAM$GA$max_terminales)) PARAM$GA$max_terminales else 60
if (length(candidatas_GA) > MAX_TERMINALES) {
  set.seed(PARAM$semilla_primigenia)
  candidatas_GA <- sample(candidatas_GA, MAX_TERMINALES)
}

cat("Variables candidatas para Grammatical Evolution:", length(candidatas_GA), "\n")

# 2. Split Local de Validación (sin tocar 202107 ni 202109)
meses_disponibles <- sort(unique(dataset$foto_mes[dataset$foto_mes < 202107]))
meses_val_GA <- tail(meses_disponibles, 3)
meses_tr_GA  <- setdiff(meses_disponibles, meses_val_GA)

idx_tr_all <- which(dataset$foto_mes %in% meses_tr_GA)
idx_val_all <- which(dataset$foto_mes %in% meses_val_GA)

# Subsampling controlado para evaluación de fitness ultrarrápida
set.seed(PARAM$semilla_primigenia)
n_fit <- if (!is.null(PARAM$GA$max_filas_fitness)) PARAM$GA$max_filas_fitness else 50000
idx_tr_GA <- if (length(idx_tr_all) > n_fit) sample(idx_tr_all, n_fit) else idx_tr_all
idx_val_GA <- if (length(idx_val_all) > (n_fit / 2)) sample(idx_val_all, n_fit / 2) else idx_val_all

y_tr_GA <- ifelse(dataset$clase_ternaria[idx_tr_GA] %in% c("BAJA+1", "BAJA+2"), 1L, 0L)
y_val_GA <- ifelse(dataset$clase_ternaria[idx_val_GA] %in% c("BAJA+1", "BAJA+2"), 1L, 0L)

# 3. Operadores Protegidos
protected_div <- function(x, y) {
  res <- x / (y + 1e-5)
  res[is.na(res) | is.infinite(res)] <- 0
  res
}

protected_log_diff <- function(x, y) {
  res <- log(abs(x - y) + 1)
  res[is.na(res) | is.infinite(res)] <- 0
  res
}

es_expresion_trivial <- function(f) {
  !grepl("[+*/-]|protected_", f)
}

# 4. Definición de la Gramática BNF
string_vars <- paste(candidatas_GA, collapse = " | ")
rule_text <- paste0(
  "<expr> ::= <op>\n",
  "<op>   ::= <op> + <op> | <op> - <op> | <op> * <op> | ",
  "protected_div(<op>, <op>) | protected_log_diff(<op>, <op>) | <var>\n",
  "<var>  ::= ", string_vars
)

tf <- tempfile()
writeLines(rule_text, tf)
bnf_grammar <- CreateGrammar(tf)
unlink(tf)

# 5. Función de Fitness con LightGBM Univariado Real
fitness_gramEvol <- function(expr) {
  valores <- tryCatch(eval(expr, envir = dataset), error = function(e) NULL)
  if (is.null(valores)) return(1)

  val_tr <- valores[idx_tr_GA]
  val_finitos <- val_tr[is.finite(val_tr)]
  if (length(val_finitos) == 0 || length(unique(val_finitos)) <= 1) {
    return(1)
  }

  dtr_ga  <- lgb.Dataset(data = matrix(val_tr, ncol = 1), label = y_tr_GA, free_raw_data = TRUE)
  dval_ga <- lgb.Dataset(data = matrix(valores[idx_val_GA], ncol = 1), label = y_val_GA, free_raw_data = TRUE)

  modelo_ga <- tryCatch({
    lgb.train(
      params = list(objective = "binary", metric = "auc",
                    learning_rate = 0.1, num_threads = 1, verbosity = -1),
      data = dtr_ga, valids = list(valid = dval_ga),
      nrounds = 50, early_stopping_rounds = 10, verbose = -1
    )
  }, error = function(e) NULL)

  if (is.null(modelo_ga) || is.null(modelo_ga$best_score) || is.na(modelo_ga$best_score)) return(1)

  1 - modelo_ga$best_score
}

evaluar_genoma <- function(genoma) {
  expr_obj <- tryCatch(suppressWarnings(GrammarMap(genoma, bnf_grammar)), error = function(e) NULL)
  if (is.null(expr_obj) || !isTRUE(GrammarIsTerminal(expr_obj))) {
    return(list(score = 0, formula = NA_character_))
  }
  expr_lang <- tryCatch(as.expression(expr_obj), error = function(e) NULL)
  if (is.null(expr_lang) || length(expr_lang) == 0) {
    return(list(score = 0, formula = NA_character_))
  }

  expr_final  <- expr_lang[[1]]
  formula_str <- paste(deparse(expr_final, width.cutoff = 500L), collapse = " ")
  costo <- fitness_gramEvol(expr_final)
  auc   <- 1 - costo

  list(score = auc, formula = formula_str)
}

# 6. Ejecución del Algoritmo Genético
set.seed(PARAM$semilla_primigenia)

cat("\n===================================================================\n")
cat(">>> INICIANDO GRAMMATICAL EVOLUTION (gramEvol) <<<\n")
cat("===================================================================\n")

ge_res <- GrammaticalEvolution(
  grammarDef      = bnf_grammar,
  evalFunc        = fitness_gramEvol,
  popSize         = PARAM$GA$popSize,
  iterations      = PARAM$GA$iterations,
  terminationCost = 0.10,
  seqLen          = PARAM$GA$seqLen,
  max.depth       = PARAM$GA$max.depth,

  # Nuevos parámetros de presión selectiva
  elitism         = as.integer(PARAM$GA$popSize * 0.50),
  mutationChance  = 0.15,

  monitorFunc     = function(result) {
    cat(sprintf("Gen %2d | Mejor Costo: %.5f (AUC: %.5f)\n",
                result$population$currentIteration,
                result$best$cost,
                1 - result$best$cost))
  }
)

# 7. Extracción e Inyección del Top N al Dataset
cat("\n=== Evaluando población final para extraer el Top", PARAM$GA$top_features, "===\n")

pop_matrix      <- ge_res$population$population
poblacion_final <- split(pop_matrix, row(pop_matrix))

n_cores <- max(1, detectCores() - 1)
resultados <- if (.Platform$OS.type == "unix") {
  mclapply(poblacion_final, evaluar_genoma, mc.cores = n_cores)
} else {
  lapply(poblacion_final, evaluar_genoma)
}

scores_finales   <- sapply(resultados, function(r) r$score)
formulas_finales <- sapply(resultados, function(r) r$formula)

ordenados <- order(scores_finales, decreasing = TRUE)

formulas_vistas <- character()
ga_cols_creadas <- character()
top_guardados   <- 0
idx             <- 1

dt_trazabilidad <- data.table(Variable=character(), AUC=numeric(), Formula=character())

while (top_guardados < PARAM$GA$top_features && idx <= length(ordenados)) {
  i <- ordenados[idx]
  idx <- idx + 1

  if (is.na(scores_finales[i]) || scores_finales[i] <= 0.50) next
  if (is.na(formulas_finales[i]) || formulas_finales[i] %in% formulas_vistas) next
  if (es_expresion_trivial(formulas_finales[i])) next

  eval_res <- tryCatch(
    eval(parse(text = formulas_finales[i])[[1]], envir = dataset),
    error = function(e) NULL
  )
  if (is.null(eval_res) || length(unique(eval_res[is.finite(eval_res)])) <= 1) next

  top_guardados <- top_guardados + 1
  formulas_vistas <- c(formulas_vistas, formulas_finales[i])

  nombre_col <- paste0("GA_Feature_", top_guardados)
  ga_cols_creadas <- c(ga_cols_creadas, nombre_col)

  cat(sprintf("[%s] AUC Univariado: %.5f | Fórmula: %s\n", nombre_col, scores_finales[i], formulas_finales[i]))
  dataset[, (nombre_col) := eval_res]
  dt_trazabilidad <- rbind(dt_trazabilidad, list(nombre_col, scores_finales[i], formulas_finales[i]))
}

fwrite(dt_trazabilidad, file = file.path(dir_experimento_base, paste0("GA_trazabilidad_formulas_s", PARAM$semilla_primigenia, ".csv")), sep = ",")
cat("\nArchivo de trazabilidad guardado en: GA_trazabilidad_formulas_s", PARAM$semilla_primigenia, ".csv\n")

cat("\nColumnas generadas por Algoritmo Genético e inyectadas al dataset:", paste(ga_cols_creadas, collapse = ", "), "\n")

# Blindaje anti-leakage: asegura que clase01 no quede en dataset antes de FEhist
if ("clase01" %in% colnames(dataset)) dataset[, clase01 := NULL]


Variables candidatas para Grammatical Evolution: 29 

>>> INICIANDO GRAMMATICAL EVOLUTION (gramEvol) <<<
Gen  1 | Mejor Costo: 0.14925 (AUC: 0.85075)
Gen  2 | Mejor Costo: 0.14925 (AUC: 0.85075)
Gen  3 | Mejor Costo: 0.14925 (AUC: 0.85075)
Gen  4 | Mejor Costo: 0.14925 (AUC: 0.85075)
Gen  5 | Mejor Costo: 0.14925 (AUC: 0.85075)
Gen  6 | Mejor Costo: 0.14925 (AUC: 0.85075)
Gen  7 | Mejor Costo: 0.14925 (AUC: 0.85075)
Gen  8 | Mejor Costo: 0.14925 (AUC: 0.85075)
Gen  9 | Mejor Costo: 0.14925 (AUC: 0.85075)
Gen 10 | Mejor Costo: 0.14925 (AUC: 0.85075)
Gen 11 | Mejor Costo: 0.13842 (AUC: 0.86158)
Gen 12 | Mejor Costo: 0.13842 (AUC: 0.86158)
Gen 13 | Mejor Costo: 0.13842 (AUC: 0.86158)
Gen 14 | Mejor Costo: 0.13842 (AUC: 0.86158)
Gen 15 | Mejor Costo: 0.13842 (AUC: 0.86158)
Gen 16 | Mejor Costo: 0.13842 (AUC: 0.86158)
Gen 17 | Mejor Costo: 0.13842 (AUC: 0.86158)
Gen 18 | Mejor Costo: 0.13842 (AUC: 0.86158)
Gen 19 | Mejor Costo: 0.13842 (AUC: 0.86158)
Gen 20 | Mejor Costo: 0.13842 (AUC: 0.86

In [41]:
# visualizo las columas del dataset a esta etapa

# Auditoría de resultados del Algoritmo Genético e inspección de columnas
cat("Total individuos en población final:", length(scores_finales), "\n")
cat("Individuos con score > 0.50:", sum(scores_finales > 0.50, na.rm = TRUE), "\n")
cat("Expresiones no triviales con score > 0.50:", sum(scores_finales > 0.50 & !sapply(formulas_finales, es_expresion_trivial), na.rm = TRUE), "\n")
cat("Expresiones únicas no triviales:", length(unique(formulas_finales[scores_finales > 0.50 & !sapply(formulas_finales, es_expresion_trivial)])), "\n")
cat("\nColumnas actuales del dataset tras Feature Engineering Intra-mes (Manual + GA):\n")
colnames(dataset)

Total individuos en población final: 151 
Individuos con score > 0.50: 25 
Expresiones no triviales con score > 0.50: 4 
Expresiones únicas no triviales: 4 

Columnas actuales del dataset tras Feature Engineering Intra-mes (Manual + GA):


[1] "numero_de_cliente"           "foto_mes"                   
 [3] "internet"                    "cliente_edad"               
 [5] "cliente_antiguedad"          "mrentabilidad"              
 [7] "mrentabilidad_annual"        "mcomisiones"                
 [9] "mactivos_margen"             "mpasivos_margen"            
[11] "cproductos"                  "mcuenta_corriente"          
[13] "mcaja_ahorro"                "cdescubierto_preacordado"   
[15] "mcuentas_saldo"              "ctarjeta_visa_transacciones"
[17] "mtarjeta_visa_consumo"       "mtarjeta_master_consumo"    
[19] "mprestamos_personales"       "cpayroll_trx"               
[21] "mpayroll"                    "ccomisiones_mantenimiento"  
[23] "ccallcenter_transacciones"   "chomebanking_transacciones" 
[25] "ctrx_quarter"                "Master_status"              
[27] "Master_fechaalta"            "Master_mpagominimo"         
[29] "Visa_status"                 "Visa_fechaalta"             
[31] "Visa_mpagominimo"            "clase_ternaria"             
[33] "kmes"                        "mpayroll_sobre_edad"        
[35] "GA_Feature_1"                "GA_Feature_2"               
[37] "GA_Feature_3"                "GA_Feature_4"

#### 6.3.1.4  FE_rf Feature Engineering de nuevas variables a partir de hojas de Random Forest

Esto se mostrará unicamente a la *modalidad Analista Sr*

In [42]:
# No se implementa Feature Engineering a partir de Random Forest

#### 6.3.1.5  FEhist Feature Engineering historico

El Fature Engineering Histórico es la etapa que más aporta a la ganancia final, ya que enriquece cada registro del dataset con su historia.

Para cada campo del dataset original (*)
se crean lo siguientes campos de a partir de la historia
* lag1  lags de orden 1
* delta1  =  valor actual - lag1
* lag2  lags de orden 2
* delta2  = valor actual - lag2


(*) Excepto para los campos  <numero_de_cliente,  foto_mes,  clase_ternaria>

In [43]:
# Problema 5 - Feature Engineering Historico
# Se opta por la recomendación del grupo A: Agregar los campos Lag 1 y 2, delta 1 y 2
# todo es lagueable, menos la primary key y la clase
cols_lagueables <- copy( setdiff(
    colnames(dataset),
    c("numero_de_cliente", "foto_mes", "clase_ternaria")
) )

# https://rdrr.io/cran/data.table/man/shift.html

# lags de orden 1
dataset[,
    paste0(cols_lagueables, "_lag1") := shift(.SD, 1, NA, "lag"),
    by = numero_de_cliente,
    .SDcols = cols_lagueables
]

# lags de orden 2
dataset[,
    paste0(cols_lagueables, "_lag2") := shift(.SD, 2, NA, "lag"),
    by = numero_de_cliente,
    .SDcols = cols_lagueables
]

# agrego los delta lags
for (vcol in cols_lagueables)
{
    dataset[, paste0(vcol, "_delta1") := get(vcol) - get(paste0(vcol, "_lag1"))]
    dataset[, paste0(vcol, "_delta2") := get(vcol) - get(paste0(vcol, "_lag2"))]
}


Verificacion de los campos recien creados

In [44]:
ncol(dataset)
colnames(dataset)

[1] 178

[1] "numero_de_cliente"                  "foto_mes"                          
  [3] "internet"                           "cliente_edad"                      
  [5] "cliente_antiguedad"                 "mrentabilidad"                     
  [7] "mrentabilidad_annual"               "mcomisiones"                       
  [9] "mactivos_margen"                    "mpasivos_margen"                   
 [11] "cproductos"                         "mcuenta_corriente"                 
 [13] "mcaja_ahorro"                       "cdescubierto_preacordado"          
 [15] "mcuentas_saldo"                     "ctarjeta_visa_transacciones"       
 [17] "mtarjeta_visa_consumo"              "mtarjeta_master_consumo"           
 [19] "mprestamos_personales"              "cpayroll_trx"                      
 [21] "mpayroll"                           "ccomisiones_mantenimiento"         
 [23] "ccallcenter_transacciones"          "chomebanking_transacciones"        
 [25] "ctrx_quarter"                       "Master_status"                     
 [27] "Master_fechaalta"                   "Master_mpagominimo"                
 [29] "Visa_status"                        "Visa_fechaalta"                    
 [31] "Visa_mpagominimo"                   "clase_ternaria"                    
 [33] "kmes"                               "mpayroll_sobre_edad"               
 [35] "GA_Feature_1"                       "GA_Feature_2"                      
 [37] "GA_Feature_3"                       "GA_Feature_4"                      
 [39] "internet_lag1"                      "cliente_edad_lag1"                 
 [41] "cliente_antiguedad_lag1"            "mrentabilidad_lag1"                
 [43] "mrentabilidad_annual_lag1"          "mcomisiones_lag1"                  
 [45] "mactivos_margen_lag1"               "mpasivos_margen_lag1"              
 [47] "cproductos_lag1"                    "mcuenta_corriente_lag1"            
 [49] "mcaja_ahorro_lag1"                  "cdescubierto_preacordado_lag1"     
 [51] "mcuentas_saldo_lag1"                "ctarjeta_visa_transacciones_lag1"  
 [53] "mtarjeta_visa_consumo_lag1"         "mtarjeta_master_consumo_lag1"      
 [55] "mprestamos_personales_lag1"         "cpayroll_trx_lag1"                 
 [57] "mpayroll_lag1"                      "ccomisiones_mantenimiento_lag1"    
 [59] "ccallcenter_transacciones_lag1"     "chomebanking_transacciones_lag1"   
 [61] "ctrx_quarter_lag1"                  "Master_status_lag1"                
 [63] "Master_fechaalta_lag1"              "Master_mpagominimo_lag1"           
 [65] "Visa_status_lag1"                   "Visa_fechaalta_lag1"               
 [67] "Visa_mpagominimo_lag1"              "kmes_lag1"                         
 [69] "mpayroll_sobre_edad_lag1"           "GA_Feature_1_lag1"                 
 [71] "GA_Feature_2_lag1"                  "GA_Feature_3_lag1"                 
 [73] "GA_Feature_4_lag1"                  "internet_lag2"                     
 [75] "cliente_edad_lag2"                  "cliente_antiguedad_lag2"           
 [77] "mrentabilidad_lag2"                 "mrentabilidad_annual_lag2"         
 [79] "mcomisiones_lag2"                   "mactivos_margen_lag2"              
 [81] "mpasivos_margen_lag2"               "cproductos_lag2"                   
 [83] "mcuenta_corriente_lag2"             "mcaja_ahorro_lag2"                 
 [85] "cdescubierto_preacordado_lag2"      "mcuentas_saldo_lag2"               
 [87] "ctarjeta_visa_transacciones_lag2"   "mtarjeta_visa_consumo_lag2"        
 [89] "mtarjeta_master_consumo_lag2"       "mprestamos_personales_lag2"        
 [91] "cpayroll_trx_lag2"                  "mpayroll_lag2"                     
 [93] "ccomisiones_mantenimiento_lag2"     "ccallcenter_transacciones_lag2"    
 [95] "chomebanking_transacciones_lag2"    "ctrx_quarter_lag2"                 
 [97] "Master_status_lag2"                 "Master_fechaalta_lag2"             
 [99] "Master_mpagominimo_lag2"            "Visa_status_lag2"                  
[1

#### 6.3.1.6  FEhist Reduccion dimensionalidad con canaritos

Esta etapa solo se mostrará a la *modalidad Anlista Sr* por algun canal secreto de forma de no confundir a los *Analista Jr*  nni distraer con detalles operativos a la estratégica *Modalidad Gerencial*

In [45]:
# No se implementa la reduccion de la dimensionalidad con canaritos

### 6.3.2 Modelado

#### 6.3.2.1 Training Strategy

Esta etapa de Workflow de  Training Strategy esta pensada para la *Modalidad Gerencial* que posee el dataset reducido de [202005, 202109]
<br> Si usted es un Analista, posee el periodo de [201901, 202109] y deberá experimentar en que meses le conviene experimentar

<br> A la *Modalidad Gerencial* no se le complicada la vida con el undersampling de los continua, por eso PARAM$trainingstrategy$training_pct <- 1.0
<br> Sin embargo, si usted es  *Analista SR* posee un dataset 50 veces ( filas x columnas) más grande que la *Modalidad Gerencial*  y por un tema de velocidad y experimentación más rápida puede llegar a necesitar activar el undersampling de la clase mayoritaria, a pesar de estar corriendo en Google Cloud.

Se hace una estrategia de entrenamiento muy sencilla, tomando todos los meses posibles, SIN eliminar nada x pandemia ni por ningun otro motivo

* future = 202109  obviamente completo

* final_train =  [ 202005, 202107 ]  SIN undersampling

* training
   * testing = NO HAY
   * validation =  202107   completo, sin undersampling
   * training = [ 202005, 202106 ]  donde se consideran el 100% de los CONTINUA

In [46]:
PARAM$trainingstrategy$validate <- c(202107)

PARAM$trainingstrategy$training <- c(
  202106, 202105, 202104, 202103, 202102, 202101,
  202012, 202011, 202010, 202009, 202008, 202007,
  202006, 202005
)

PARAM$trainingstrategy$training_pct <- 1.0


PARAM$trainingstrategy$positivos <- c( "BAJA+1", "BAJA+2")

In [47]:
# seteo la clase01   1={BAJA+1, BAJA+2}   0={CONTINUA}
dataset[, clase01 := ifelse( clase_ternaria %in% PARAM$trainingstrategy$positivos, 1, 0 )]

In [48]:
# los campos en los que se entrena
campos_buenos <- copy( setdiff(
    colnames(dataset), c("clase_ternaria","clase01","azar"))
)

Esta celda tarda en correr interminables 7 minutos en Colab
<br> ya que debe instalar la librería de LightGBM

In [49]:
# preparo para que se puede hacer undersampling de los CONTINUA
#  solamente por un tema de VELOCIDAD
set.seed(PARAM$semilla_primigenia, kind = "L'Ecuyer-CMRG")
dataset[, azar:=runif(nrow(dataset))]

# undersampling de los CONTINUA
dataset[, fold_train :=  foto_mes %in%  PARAM$trainingstrategy$training &
    (clase_ternaria %in% c("BAJA+1", "BAJA+2") |
     azar < PARAM$trainingstrategy$training_pct ) ]


if( !require("lightgbm")) install.packages("lightgbm")
require("lightgbm")

dtrain <- lgb.Dataset(
  data= data.matrix(dataset[fold_train == TRUE, campos_buenos, with = FALSE]),
  label= dataset[fold_train == TRUE, clase01],
  free_raw_data= TRUE
)

In [50]:
# datos de validation
dvalidate <- lgb.Dataset(
  data= data.matrix(dataset[foto_mes %in% PARAM$trainingstrategy$validate, campos_buenos, with = FALSE]),
  label= dataset[foto_mes %in% PARAM$trainingstrategy$validate, clase01],
  free_raw_data= TRUE
)

nrow(dvalidate)

[1] 13202

####  6.3.2.2. Hyperparameter Tuning

* Clase binaria que se optimiza :  positivos = [ BAJA+1, BAJA+2 ]

* Metrica que se optimiza **AUC** Area Under Curve de la  ROC Curve

es muy importante notar que intencionalmente  **NO** se está optimizando la funcion de ganancia del problema

* Parametros no default, fijos de LightGBM que no se optimizan
  * max_bin = 31 , Alienigenas Ancestrales contruyeron las pirámides y dejaron a la humanidad en un jeroglifico  *max_bin=31*
  * feature_fraction = 0.5  para poner algo que generalmente no falla
  * learning_rate = 0.03  para que aprenda lento


* Parametros que se optimizan en el Grid Search
  * num_leaves  [64, 512]
  * min_data_in_leaf  [64, 2048]

In [51]:
# parametros fijos del LightGBM
PARAM$lgbm$param_fijos <- list(
  objective= "binary",
  metric= "auc",
  first_metric_only= TRUE,
  boost_from_average= TRUE,
  feature_pre_filter= FALSE,
  verbosity= -100,
  force_row_wise= TRUE, # para evitar warning
  seed= PARAM$semilla_primigenia,
  max_bin= 31,
  learning_rate= 0.03,
  feature_fraction= 0.5,
  num_iterations= 2048,  # valor grande, lo limita early_stopping_rounds
  early_stopping_rounds= 200,
  num_leaves= 64,
  min_data_in_leaf= 128
)


In [52]:
# En  x llegan los parametros moviles de LightGBM
#  devuelve la AUC en validate del modelo entrenado
#  en el parametro x llegan los hiperparámetros que se estan optimizando

Estimar_AUC_lightgbm <- function(x) {

  # x pisa (o agrega) a param_fijos
  param_completo <- modifyList(PARAM$lgbm$param_fijos, x)

  # entreno LightGBM
  modelo_train <- lgb.train(
    data= dtrain,
    valids= list(valid = dvalidate),
    eval= "auc",
    param= param_completo,
    verbose= -100
  )

  # recupero la AUC en validation
  AUC <- modelo_train$record_evals$valid$auc$eval[[modelo_train$best_iter]]

  message(format(Sys.time(), "%a %b %d %X %Y  "),
    toString(x),
    " niter ", modelo_train$best_iter,
    " AUC ", AUC
  )

  niter <- modelo_train$best_iter
  # hago espacio en la memoria
  rm(modelo_train)
  gc(full= TRUE, verbose= FALSE)

  return( list(AUC, niter))
}

seteo del Grid Search

In [53]:
# lo que sigue a continuacion es una forma alternativa a los loops anidados
# creo una tabla con el producto cartesiano de los vectores
tb_nueva <- CJ(
  num_leaves= c(64, 128, 256, 512),
  min_data_in_leaf= c(64, 256, 512, 1024, 2048)
)

Corrida del Grid Search,  aqui se hace el trabajo pesado
<br> por favor no se asuste con los warnings que pudieran aparecer
<br> ATENCION, la siguiente celda demora 50 minutos en Colab
<br> lamento profundamente tal intolerable espera gerencial

In [54]:
# registro a registro calculo la AUC
tb_nueva[, c("AUC", "num_iterations"):= Estimar_AUC_lightgbm( .SD ),
  by=1:nrow(tb_nueva) ]

Thu Sep 17 12:54:38 AM 2026  64, 64 niter 499 AUC 0.947872084322847

Thu Sep 17 12:56:15 AM 2026  64, 256 niter 459 AUC 0.948768665213572

Thu Sep 17 12:59:43 AM 2026  64, 512 niter 1204 AUC 0.948355726167626

Thu Sep 17 01:00:54 AM 2026  64, 1024 niter 243 AUC 0.948005248489975

Thu Sep 17 01:03:36 AM 2026  64, 2048 niter 799 AUC 0.951192947070497

Thu Sep 17 01:05:14 AM 2026  128, 64 niter 384 AUC 0.950116789745925

Thu Sep 17 01:06:35 AM 2026  128, 256 niter 268 AUC 0.949815760651507

Thu Sep 17 01:09:05 AM 2026  128, 512 niter 666 AUC 0.951751195549628

Thu Sep 17 01:11:51 AM 2026  128, 1024 niter 724 AUC 0.949264452324409

Thu Sep 17 01:17:28 AM 2026  128, 2048 niter 1866 AUC 0.953045967663229

Thu Sep 17 01:20:03 AM 2026  256, 64 niter 604 AUC 0.951350835529241

Thu Sep 17 01:22:22 AM 2026  256, 256 niter 486 AUC 0.954169404773523

Thu Sep 17 01:25:32 AM 2026  256, 512 niter 748 AUC 0.951244130691738

Thu Sep 17 01:27:09 AM 2026  256, 1024 niter 317 AUC 0.948349653534598

Thu Sep

la optimizacion de hiperparámetros de tipo  Grid Search ha corrido, extraigo los mejores hiperparametros

In [55]:
tb_nueva

fwrite( tb_nueva,
  file= "tb_grid_search_01.txt",
  sep="\t",
  append= TRUE
)

num_leaves,min_data_in_leaf,AUC,num_iterations
<dbl>,<dbl>,<dbl>,<int>
64,64,0.9478721,499
64,256,0.9487687,459
64,512,0.9483557,1204
64,1024,0.9480052,243
64,2048,0.9511929,799
128,64,0.9501168,384
128,256,0.9498158,268
128,512,0.9517512,666
128,1024,0.9492645,724


In [56]:
setorder( tb_nueva, -AUC)  # ordeno DESCENDENTE por AUC
PARAM$out$lgbm$AUC <- tb_nueva[1, AUC] # en la posicion 1 estan los mejores
PARAM$out$lgbm$mejores_hiperparametros <- as.list( tb_nueva[1] )
PARAM$out$lgbm$mejores_hiperparametros$AUC <- NULL
PARAM$out$lgbm$mejores_hiperparametros

$num_leaves
[1] 256

$min_data_in_leaf
[1] 256

$num_iterations
[1] 486

### 6.3.3 Produccion

#### Final Training
Construyo el modelo final, que es uno solo, no hace ningun tipo de particion < training, validation, testing>]

##### Final Training Dataset

Aqui esta la gran decision de en qué meses hago el Final Training
<br> debo utilizar los mejores hiperparámetros que encontré en optimización de hiperparámetros

In [57]:
PARAM$trainingstrategy$final_train <- c( 202107,
  202106, 202105, 202104, 202103, 202102, 202101,
  202012, 202011, 202010, 202009, 202008, 202007,
  202006, 202005
)

dataset[, fold_final_train := foto_mes %in% PARAM$trainingstrategy$final_train ]

# creo el dfinal_train en formato  LightGBM
dfinal_train <- lgb.Dataset(
  data= data.matrix(dataset[fold_final_train == TRUE, campos_buenos, with= FALSE]),
  label= dataset[fold_final_train == TRUE, clase01],
  free_raw_data= TRUE
)

nrow( dfinal_train) # verifico el tamaño

[1] 192651

##### Final Training Hyperparameters

In [58]:
# uno los parametros fijos y los mejores encontrados de los variables
fijos <- copy(PARAM$lgbm$param_fijos)

# quito lo que optimice en la Bayesian Optimization
fijos$num_iterations <- NULL
fijos$early_stopping_rounds <- NULL

# agrego a los hiperparametros fijos los que encontre con la Bayesian Optimization
param_final <- c(fijos, PARAM$out$lgbm$mejores_hiperparametros)

##### Training
Genero el modelo final, siempre sobre TODOS los datos de  final_train, sin hacer ningun tipo de undersampling de la clase mayoritaria

In [59]:
final_model <- lgb.train(
  data= dfinal_train,
  param= param_final,
  verbose= -100
)

In [60]:
# grabo a disco el modelo en un formato para seres humanos ... ponele ...

lgb.save(final_model, "modelo.txt")

In [61]:
# ahora imprimo la importancia de variables

tb_importancia <- as.data.table(lgb.importance(final_model))
archivo_importancia <- "impo.txt"

fwrite( tb_importancia,
  file= archivo_importancia,
  sep= "\t"
)

#### Scoring

Aplico el modelo final a los datos del futuro

In [62]:
PARAM$trainingstrategy$future <- c(202109)

dfuture <- dataset[ foto_mes %in% PARAM$trainingstrategy$future ]

In [63]:
# aplico final_model   a dfuture

prediccion <- predict(
  final_model,
  data.matrix(dfuture[, campos_buenos, with= FALSE])
)

##### Tabla Prediccion

In [64]:
tb_prediccion <- dfuture[, list(numero_de_cliente)]
tb_prediccion[, prob := prediccion]

# grabo las probabilidad del modelo
#  me va a ser util para hacer Ensembles de modelos
fwrite(tb_prediccion,
  file= "prediccion.txt",
  sep= "\t"
)

#### Kaggle Competition Submit

Genero las salidas y hago los submits a Kaggle
<br>El notebook esta preparado para la Modalidad Gerencial, los analistas deben hacer cambios.


In [65]:
# genero archivos con los  "envios" mejores
# suba TODOS los archivos a Kaggle

PARAM$kaggle$competencia <- "utn-2026-virtual-mgr"
PARAM$kaggle$cortes <- seq(800, 1300, by = 50)

# ordeno por probabilidad descendente
setorder(tb_prediccion, -prob)

dir.create("kaggle")

for (envios in PARAM$kaggle$cortes) {

  tb_prediccion[, Predicted := 0L] # seteo inicial a 0
  tb_prediccion[1:envios, Predicted := 1L] # marclo los primeros

  archivo_kaggle <- paste0("./kaggle/KA", PARAM$experimento, "_", envios, ".csv")

  # grabo el archivo
  fwrite(tb_prediccion[, list(numero_de_cliente, Predicted)],
    file= archivo_kaggle,
    sep= ","
  )

  # subida a Kaggle, armo la linea de comando
  comando <- "kaggle competitions submit"
  competencia <- paste("-c", PARAM$kaggle$competencia)
  arch <- paste( "-f", archivo_kaggle)

  mensaje <- paste0("-m 'envios=", envios,
  "  semilla=", PARAM$semilla_primigenia,
    "'" )

  linea <- paste( comando, competencia, arch, mensaje)

  Sys.sleep(30)
  salida <- system(linea, intern=TRUE) # el submit a Kaggle
  cat(salida, "\n")
}

In [66]:
# grabo los parametros
if( !require("yaml")) install.packages("yaml")
require("yaml")

write_yaml( PARAM, file="PARAM.yml")

Loading required package: yaml



In [67]:
format(Sys.time(), "%a %b %d %X %Y")

[1] "Thu Sep 17 01:52:54 AM 2026"